# Feature selection pipeline runs (derived_8.4-feature-selection-1.0)

This notebook drives feature selection runs across variants C0–C5 on the `derived_8.4` dataset splits. Selection uses train only (val/test are passed for internal metrics). Artifacts land under `artifacts/derived_8.4/<variant>/selected_features.json`.

In [1]:
from pathlib import Path
import sys
import json
import subprocess

PROJECT_ROOT = Path.cwd().resolve()
for p in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (p / "data" / "splits").is_dir() and (p / "Modeling").is_dir():
        PROJECT_ROOT = p
        break
sys.path.insert(0, str(PROJECT_ROOT))

EXP_DIR = PROJECT_ROOT / "notebooks" / "experiment" / "derived_8.4-feature-selection-1.0"
RUNNER = EXP_DIR / "run_selection.py"
assert RUNNER.exists(), RUNNER
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUNNER:", RUNNER)

PROJECT_ROOT: /scratch/user/u.rp352032/MDR-Project
RUNNER: /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.4-feature-selection-1.0/run_selection.py


## Smoke test (c2_xgb variant)

Run the primary `c2_xgb` variant with few bootstraps to validate wiring before running the full sweep.

In [2]:
cmd = [
    sys.executable, str(RUNNER),
    "--variants", "c2_xgb",
    "--dataset", "derived_8.4",
    "--n-boot", "8",
]
print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True)
print(proc.stdout[-4000:] if proc.stdout else "")
if proc.returncode != 0:
    print(proc.stderr[-4000:])
    raise RuntimeError(f"run_selection smoke test failed with code {proc.returncode}")
print("Smoke test OK")

Running: /scratch/user/u.rp352032/MDR-Project/notebooks/.venv/bin/python3 /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.4-feature-selection-1.0/run_selection.py --variants c2_xgb --dataset derived_8.4 --n-boot 8


[derived_8.4/c2_xgb] selected 50 features → /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.4-feature-selection-1.0/artifacts/derived_8.4/c2_xgb/selected_features.json
Summary written to /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.4-feature-selection-1.0/artifacts/run_summary.json

Smoke test OK


## Full ablation sweep

Runs all feature selection variants C0–C5 on dataset `derived_8.4` with default stability bootstraps.

In [3]:
RUN_FULL = False

if RUN_FULL:
    cmd = [sys.executable, str(RUNNER), "--dataset", "derived_8.4"]
    print("Running full sweep on derived_8.4...")
    proc = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True)
    print(proc.stdout[-6000:] if proc.stdout else "")
    if proc.returncode != 0:
        print(proc.stderr[-6000:])
        raise RuntimeError(f"full sweep failed: {proc.returncode}")
    print("Full sweep complete")
else:
    print("Skipped full sweep (RUN_FULL=False)")

Skipped full sweep (RUN_FULL=False)


## Artifact summary

Inspect family counts and feature lists for every produced selection on `derived_8.4`.

In [4]:
import json
from pathlib import Path
import pandas as pd
from Modeling.Src.soilmoist_fl.Selectors.family_coverage import group_by_coverage_family

rows = []
art_root = EXP_DIR / "artifacts"
for path in sorted(art_root.glob("derived_8.4/*/selected_features.json")):
    payload = json.loads(path.read_text())
    feats = payload["features"]
    groups = group_by_coverage_family(feats)
    rows.append({
        "dataset": payload["dataset"],
        "variant": payload["variant"],
        "n": payload["n_features"],
        "satellite": len(groups.get("satellite", [])),
        "hydro": len(groups.get("hydro", [])),
        "static": len(groups.get("static", [])),
        "calendar": len(groups.get("calendar", [])),
        "temporal": len(groups.get("temporal", [])),
        "path": str(path.relative_to(PROJECT_ROOT)),
    })

summary = pd.DataFrame(rows)
if len(summary):
    print(summary.to_string(index=False))
    summary.to_csv(art_root / "artifact_summary.csv", index=False)
else:
    print("No artifacts found yet.")

    dataset                variant  n  satellite  hydro  static  calendar  temporal                                                                                                                       path
derived_8.4  c0_baseline_bypass_on 46         24      6       5        10         0  notebooks/experiment/derived_8.4-feature-selection-1.0/artifacts/derived_8.4/c0_baseline_bypass_on/selected_features.json
derived_8.4 c1_baseline_bypass_off 12         11      0       0         1         0 notebooks/experiment/derived_8.4-feature-selection-1.0/artifacts/derived_8.4/c1_baseline_bypass_off/selected_features.json
derived_8.4                 c2_xgb 50         28      7       7         8         0                 notebooks/experiment/derived_8.4-feature-selection-1.0/artifacts/derived_8.4/c2_xgb/selected_features.json
derived_8.4       c2b_xgb_softcorr 55         32     12       4         7         0       notebooks/experiment/derived_8.4-feature-selection-1.0/artifacts/derived_8.4/c2b_x